In [ ]:
import os
import copy
import random
import numpy as np
import pandas as pd

from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix


# Load the old CNN

In [ ]:
# ── CUDA (RTX 4060 Ti) setup ──────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device("cuda")
    # TF32 acelera matmul en Ampere+ sin pérdida práctica de precisión
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True   # optimiza kernels para tamaño fijo de input
    print("Usando dispositivo:", device)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM total:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    device = torch.device("cpu")
    print("CUDA no disponible, usando CPU")

# Load the model and configure for the device
model_path = "Models/Model_Exec_711_Epoch_150_cpu_portable_full.pt"

values = torch.load(model_path, map_location="cpu", weights_only=False)
model = values["modelo"]

model = model.to(device)
model.eval()

print("Modelo cargado correctamente")


# Capture the tensor and get the vector

In [ ]:
#Tensor from model to vector[1024] extracter:

class CNNEmbeddingExtractor(nn.Module):
    def __init__(self, base_model):
        super().__init__()

        self.features = nn.ModuleList([
            nn.Sequential(*copy.deepcopy(block))
            for block in base_model.features
        ])

        self.second_level = copy.deepcopy(base_model.second_level)
        self.sizes = copy.deepcopy(base_model.sizes)

    def forward(self, image):
        x = image
        prev = -1
        pos = 0
        outputs = {}

        for i in range(len(self.features)):
            if i == 0 or i == 1:
                x = self.features[i](x)
                outputs[i] = x
            else:
                connections = self.second_level[pos:pos+prev]

                for c in range(len(connections)):
                    if connections[c] == 1:
                        skip_size = self.sizes[c][0]
                        req_size = x.shape[2]

                        if skip_size > req_size:
                            psize = skip_size - req_size + 1
                            pool = nn.MaxPool2d(kernel_size=psize, stride=1)
                            x2 = pool(outputs[c])

                        elif skip_size == req_size:
                            x2 = outputs[c]

                        else:
                            if (req_size - skip_size) % 2 == 0:
                                pad = int((req_size - skip_size) / 2)
                                padding = nn.ZeroPad2d(pad)
                                x2 = padding(outputs[c])
                            else:
                                pool = nn.MaxPool2d(kernel_size=2, stride=1, padding=1)
                                x2 = pool(outputs[c])
                                pad = int((req_size - skip_size - 1) / 2)
                                if pad > 0:
                                    padding = nn.ZeroPad2d(pad)
                                    x2 = padding(x2)

                        x = torch.cat((x, x2), axis=1)

                x = self.features[i](x)
                outputs[i] = x
                pos += prev

            prev += 1

        return torch.flatten(x, 1)


# Create the clinical vector

In [ ]:
#Transformer for the image

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])


In [ ]:
#Check the csv
import pandas as pd

df = pd.read_csv("DataSet/dataset_maestro_od.csv")
print(df.shape)
print(df.columns.tolist())
df.head()


In [ ]:
def build_clinical_vector(dx_periapical, sensibilidad, percusion, movilidad, palpacion, od):
    # Normalizar strings
    dx_periapical = str(dx_periapical).strip()
    sensibilidad = str(sensibilidad).strip().lower()
    percusion = str(percusion).strip().lower()
    movilidad = str(movilidad).strip()
    palpacion = str(palpacion).strip().lower()

    # Normalizar OD
    try:
        od = float(str(od).strip())
    except ValueError:
        raise ValueError(f"OD no válido: {od}")

    # Mapeo Dx Periapical
    dx_map = {
        "Periodontitis apical asintomática": [1, 0, 0, 0, 0],
        "Absceso apical crónico":            [0, 1, 0, 0, 0],
        "Periodontitis apical sintomática":  [0, 0, 1, 0, 0],
        "Tejidos apicales normales":         [0, 0, 0, 1, 0],
        "Absceso apical agudo":              [0, 0, 0, 0, 1],
    }

    # Binarias
    bin_map = {
        "positiva": 1.0,
        "positivo": 1.0,
        "negativa": 0.0,
        "negativo": 0.0,
    }

    # Movilidad
    movilidad_map = {
        "No valorable": [0, 0, 0],
        "1":            [1, 0, 0],
        "2":            [0, 1, 0],
        "3":            [0, 0, 1],
    }

    if dx_periapical not in dx_map:
        raise ValueError(f"Dx Periapical no válido: {dx_periapical}")

    if sensibilidad not in bin_map:
        raise ValueError(f"Sensibilidad no válida: {sensibilidad}")

    if percusion not in bin_map:
        raise ValueError(f"Percusión no válida: {percusion}")

    if palpacion not in bin_map:
        raise ValueError(f"Palpación no válida: {palpacion}")

    if movilidad not in movilidad_map:
        raise ValueError(f"Movilidad no válida: {movilidad}")

    vector = (
        dx_map[dx_periapical]
        + [bin_map[sensibilidad]]
        + [bin_map[percusion]]
        + movilidad_map[movilidad]
        + [bin_map[palpacion]]
        + [od]
    )

    return torch.tensor(vector, dtype=torch.float32)


In [ ]:
#This aren't the clases for the vector, this is for the final classification of the model.

class_to_idx = {
    "Necrosis Pulpar": 0,
    "Previamente iniciado": 1,
    "Previamente Tratado": 2,
    "Pulpa Normal": 3,
    "Pulpitis Irreversible Asintomática": 4,
    "Pulpitis Irreversible Sintomática": 5,
    "Pulpitis Reversible": 6,
}


In [ ]:
from pathlib import Path

def normalize_image_key(path):
    path = str(path).strip()

    # Unificar separadores
    path = path.replace("\\", "/")

    # Quitar prefijos posibles
    prefixes = [
        "DataSet/",
        "radiografia_dataset_original/",
        "DataSet/radiografia_dataset_original/",
    ]

    for prefix in prefixes:
        if path.startswith(prefix):
            path = path[len(prefix):]

    return path.strip("/")


# Data Loader

In [ ]:
# Check the images
from pathlib import Path

def build_image_path_dict(root_dir, valid_exts=None):
    if valid_exts is None:
        valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

    root_dir = Path(root_dir)
    image_path_dict = {}
    failed_files = []

    all_files = sorted(root_dir.rglob("*"))

    for file_path in all_files:
        if file_path.is_file() and file_path.suffix.lower() in valid_exts:
            try:
                relative_key = str(file_path.relative_to(root_dir))
                relative_key = normalize_image_key(relative_key)

                image_path_dict[relative_key] = str(file_path)

            except Exception as e:
                failed_files.append((str(file_path), str(e)))

    return image_path_dict, failed_files


In [ ]:
%pwd

In [ ]:
# Build the image path dictionary
root_dir = "DataSet/radiografia_dataset_original"

image_path_dict, failed_files = build_image_path_dict(
    root_dir=root_dir
)

print("Número de imágenes indexadas:", len(image_path_dict))
print("Número de archivos con error:", len(failed_files))

first_keys = list(image_path_dict.keys())[:10]

for k in first_keys:
    print("Clave:", k)
    print("Ruta:", image_path_dict[k])
    print()

if failed_files:
    print("Archivos con error:")
    for path, err in failed_files[:30]:
        print(path)
        print("Error:", err)
        print("-" * 50)
else:
    print("No hubo errores.")


In [ ]:
# Create the samples

samples = []
errors = []

for idx, row in df.iterrows():
    try:
        image_key = normalize_image_key(row["external_id"])

        if image_key not in image_path_dict:
            raise KeyError(f"No se encontró imagen para: {image_key}")

        clinical_vector = build_clinical_vector(
            dx_periapical=row["Dx Periapical"],
            sensibilidad=row["Sensibilidad"],
            percusion=row["Percusión"],
            movilidad=row["Movilidad"],
            palpacion=row["Palpación"],
            od=row["OD"],
        )

        if row["Clase"] not in class_to_idx:
            raise KeyError(f"Clase no válida: {row['Clase']}")

        label = class_to_idx[row["Clase"]]

        samples.append({
            "image_key": image_key,
            "image_path": image_path_dict[image_key],  
            "clinical_vector": clinical_vector,         # [12]
            "label": label,                             # int
        })

    except Exception as e:
        errors.append({
            "row_idx": idx,
            "id": row.get("id", None),
            "external_id": row.get("external_id", None),
            "clase": row.get("Clase", None),
            "dx_periapical": row.get("Dx Periapical", None),
            "sensibilidad": row.get("Sensibilidad", None),
            "percusion": row.get("Percusión", None),
            "movilidad": row.get("Movilidad", None),
            "palpacion": row.get("Palpación", None),
            "error": str(e),
        })

print("Samples válidos:", len(samples))
print("Errores:", len(errors))

for err in errors:
    print("=" * 80)
    print(f"Fila: {err['row_idx']}")
    print(f"id: {err['id']}")
    print(f"external_id: {err['external_id']}")
    print(f"Clase: {err['clase']}")
    print(f"Dx Periapical: {err['dx_periapical']}")
    print(f"Sensibilidad: {err['sensibilidad']}")
    print(f"Percusión: {err['percusion']}")
    print(f"Movilidad: {err['movilidad']}")
    print(f"Palpación: {err['palpacion']}")
    print(f"Error: {err['error']}")


In [ ]:
# Data set Class for the stratification
from torch.utils.data import Dataset
from PIL import Image
import torch

class FullMultimodalFolderDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform
        self.targets = [s["label"] for s in samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        image = Image.open(sample["image_path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        clinical = torch.tensor(sample["clinical_vector"], dtype=torch.float32)
        label = torch.tensor(sample["label"], dtype=torch.long)

        return image, clinical, label


In [ ]:
# make the stratified splits

import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit

def make_stratified_splits(targets, val_split=0.15, test_split=0.15, seed=42):
    y = np.array(targets)
    idx = np.arange(len(y))

    # train+val vs test
    sss_test = StratifiedShuffleSplit(
        n_splits=1,
        test_size=test_split,
        random_state=seed
    )
    trainval_idx, test_idx = next(sss_test.split(idx, y))

    # train vs val
    y_trainval = y[trainval_idx]
    val_rel = val_split / (1.0 - test_split)

    sss_val = StratifiedShuffleSplit(
        n_splits=1,
        test_size=val_rel,
        random_state=seed
    )
    train_rel, val_rel_idx = next(sss_val.split(trainval_idx, y_trainval))

    train_idx = trainval_idx[train_rel]
    val_idx = trainval_idx[val_rel_idx]

    return {
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
    }


In [ ]:
# Data loader — optimizado para CUDA (RTX 4060 Ti)

import torch
from torch.utils.data import DataLoader, Subset

def make_multimodal_dataloaders(
    ds,
    train_idx,
    val_idx,
    test_idx,
    batch_size=64,
    num_workers=4,
    seed=42,
):
    g = torch.Generator()
    g.manual_seed(seed)

    # pin_memory=True solo es útil con CUDA
    pin = device.type == "cuda"

    train_dl = DataLoader(
        Subset(ds, train_idx),
        batch_size=batch_size,
        shuffle=True,
        generator=g,
        num_workers=num_workers,
        pin_memory=pin,
        persistent_workers=(num_workers > 0),   # evita re-crear workers en cada epoch
        prefetch_factor=2 if num_workers > 0 else None,
    )

    val_dl = DataLoader(
        Subset(ds, val_idx),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin,
        persistent_workers=(num_workers > 0),
        prefetch_factor=2 if num_workers > 0 else None,
    )

    test_dl = DataLoader(
        Subset(ds, test_idx),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin,
        persistent_workers=(num_workers > 0),
        prefetch_factor=2 if num_workers > 0 else None,
    )

    return {
        "train": train_dl,
        "val": val_dl,
        "test": test_dl,
    }


In [ ]:
# Use the Data loader
def load_data_module(
    samples,
    batch_size=64,
    val_split=0.15,
    test_split=0.15,
    seed=42,
    num_workers=4    # 4 workers es óptimo para la mayoría de CPUs en Windows/Linux con RTX
):
    ds = FullMultimodalFolderDataset(
        samples=samples,
        transform=transform
    )

    splits = make_stratified_splits(
        targets=ds.targets,
        val_split=val_split,
        test_split=test_split,
        seed=seed,
    )

    loaders = make_multimodal_dataloaders(
        ds=ds,
        train_idx=splits["train_idx"],
        val_idx=splits["val_idx"],
        test_idx=splits["test_idx"],
        batch_size=batch_size,
        num_workers=num_workers,
        seed=seed,
    )

    return {
        "dataset": ds,
        "loaders": loaders,
        "splits": splits,
    }


In [ ]:
# Functions to show the distribution of classes in each split
from collections import Counter

def show_split_distribution(ds, split_idx, name, idx_to_class):
    labels = [ds.targets[i] for i in split_idx]
    counts = Counter(labels)
    total = len(labels)

    print(f"\n{name} total = {total}")
    for class_idx in sorted(idx_to_class.keys()):
        class_name = idx_to_class[class_idx]
        n = counts.get(class_idx, 0)
        pct = (n / total) * 100 if total > 0 else 0
        print(f"  {class_name:<35} {n:>3}  ({pct:>5.2f}%)")


def print_data_module_summary(data_module, class_to_idx):
    ds = data_module["dataset"]
    splits = data_module["splits"]

    train_idx = splits["train_idx"]
    val_idx = splits["val_idx"]
    test_idx = splits["test_idx"]

    idx_to_class = {v: k for k, v in class_to_idx.items()}

    print(f"Counts: {len(train_idx)} {len(val_idx)} {len(test_idx)}")

    show_split_distribution(ds, train_idx, "TRAIN", idx_to_class)
    show_split_distribution(ds, val_idx, "VAL", idx_to_class)
    show_split_distribution(ds, test_idx, "TEST", idx_to_class)

    all_idx = np.concatenate([train_idx, val_idx, test_idx])
    print("¿Índices únicos?", len(all_idx) == len(set(all_idx)))
    print("¿Algún índice fuera de rango?", max(all_idx) >= len(ds))
    


In [ ]:
#Show the distribution of classes in each split
data_module = load_data_module(samples, num_workers=4)

print_data_module_summary(data_module, class_to_idx)


# Create the new model

In [ ]:
import torch
import torch.nn as nn

class FusedClassifier(nn.Module):
    def __init__(
        self,
        input_dim=18444,
        num_classes=6,
        dropout=0.2,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2), 
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),  
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
#Create the full model

class FullMultimodalDentalDeppGAModel(nn.Module):
    def __init__(self, embedding_extractor, fused_classifier):
        super().__init__()
        self.embedding_extractor = embedding_extractor
        self.fused_classifier = fused_classifier

    def forward(self, image, clinical_vector):
        embedding = self.embedding_extractor(image)         
        x = torch.cat([embedding, clinical_vector], dim=1)  
        logits = self.fused_classifier(x)                    # [B, 7]
        return logits


In [ ]:
#Testing if the model exists

clinical_dim = len(samples[0]["clinical_vector"])
num_classes = len(class_to_idx)
embedding_dim = 18432

print("clinical_dim:", clinical_dim)
print("num_classes:", num_classes)

extractor = CNNEmbeddingExtractor(model)

fused_classifier = FusedClassifier(
    input_dim=embedding_dim + clinical_dim,
    num_classes=num_classes,
)

full_model = FullMultimodalDentalDeppGAModel(
    embedding_extractor=extractor,
    fused_classifier=fused_classifier,
).to(device)

print(full_model)


In [ ]:
#Testing if the model can process a batch of data

train_loader = data_module["loaders"]["train"]

images, clinical_vectors, labels = next(iter(train_loader))

images = images.to(device, non_blocking=True)
clinical_vectors = clinical_vectors.to(device, non_blocking=True)
labels = labels.to(device, non_blocking=True)

with torch.no_grad():
    embedding = full_model.embedding_extractor(images)
    outputs = full_model(images, clinical_vectors)

print("images:", images.shape)
print("clinical_vectors:", clinical_vectors.shape)
print("embedding:", embedding.shape)
print("labels:", labels.shape)
print("outputs:", outputs.shape)


In [ ]:
images, clinical_vectors, labels = next(iter(train_loader))
print("labels en el batch:", labels)
print("unique labels:", labels.unique())
print("clinical_vectors muestra:", clinical_vectors[0])


# Training the Full Model

In [ ]:
import torch
import torch.nn as nn
import copy

# ── Verificación del dispositivo ──────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    print("CUDA disponible:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Usando:", device)

full_model = full_model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW([
        {"params": full_model.embedding_extractor.parameters(), "lr": 1e-6},
        {"params": full_model.fused_classifier.parameters(), "lr": 3e-4},
    ], weight_decay=5e-4)

# ── Automatic Mixed Precision scaler (FP16 en CUDA) ──────────────────────────
# AMP reduce el uso de VRAM ~50% y acelera el entrenamiento en Tensor Cores
scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None
print("AMP GradScaler:", "activado" if scaler is not None else "desactivado")


In [ ]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
    scaler=None,        # GradScaler para AMP (CUDA); None deshabilita AMP
):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, clinical_vectors, labels in loader:
        # non_blocking=True solapa la transferencia CPU→GPU con el cómputo
        images = images.to(device, non_blocking=True)
        clinical_vectors = clinical_vectors.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            # FP16 forward pass dentro del autocast
            with torch.amp.autocast("cuda"):
                outputs = model(images, clinical_vectors)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images, clinical_vectors)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        preds = outputs.argmax(dim=1)

        batch_total = labels.size(0)
        running_loss += loss.item() * batch_total
        correct += (preds == labels).sum().item()
        total += batch_total

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


In [ ]:
#Test if the model can train
train_loss, train_acc = train_one_epoch(
    full_model,
    data_module["loaders"]["train"],
    criterion,
    optimizer,
    device,
    scaler=scaler,
)

print(f"train_loss = {train_loss:.4f}")
print(f"train_acc  = {train_acc:.4f}")


In [ ]:
def evaluate_one_epoch(
    model,
    loader,
    criterion,
    device,
):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.inference_mode():
        for images, clinical_vectors, labels in loader:
            images = images.to(device, non_blocking=True)
            clinical_vectors = clinical_vectors.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            # autocast también acelera inference
            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                outputs = model(images, clinical_vectors)
                loss = criterion(outputs, labels)

            preds = outputs.argmax(dim=1)

            batch_total = labels.size(0)
            running_loss += loss.item() * batch_total
            correct += (preds == labels).sum().item()
            total += batch_total

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs=50,
    min_acc_delta=0.01,
    scaler=None,        # GradScaler AMP
):
    best_model_wts = copy.deepcopy(model.state_dict())

    best_val_acc = 0.0
    best_val_loss = float("inf")
    best_epoch = 0

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            scaler=scaler,
        )

        val_loss, val_acc = evaluate_one_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        # Monitoreo de VRAM (solo CUDA)
        vram_info = ""
        if device.type == "cuda":
            used  = torch.cuda.memory_allocated(device) / 1e9
            reserved = torch.cuda.memory_reserved(device) / 1e9
            vram_info = f" | VRAM {used:.2f}/{reserved:.2f} GB"

        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f}"
            f"{vram_info}"
        )

        acc_improved = val_acc > best_val_acc + min_acc_delta
        acc_similar_but_loss_better = (
            abs(val_acc - best_val_acc) <= min_acc_delta
            and val_loss < best_val_loss
        )

        if acc_improved or acc_similar_but_loss_better:
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_epoch = epoch + 1
            best_model_wts = copy.deepcopy(model.state_dict())

            print(
                f"  ✅ Nuevo mejor modelo | "
                f"Epoch {best_epoch} | "
                f"Val Acc: {best_val_acc:.4f} | "
                f"Val Loss: {best_val_loss:.4f}"
            )

    model.load_state_dict(best_model_wts)

    print("\nMejor modelo:")
    print(f"Epoch: {best_epoch}")
    print(f"Val Acc: {best_val_acc:.4f}")
    print(f"Val Loss: {best_val_loss:.4f}")

    return model, history


In [ ]:
train_loader = data_module["loaders"]["train"]
val_loader = data_module["loaders"]["val"]

#train_full_model, history = train_model(
#    model=full_model,
#    train_loader=train_loader,
#    val_loader=val_loader,
#    criterion=criterion,
#    optimizer=optimizer,
#    device=device,
#    epochs=50,
#    scaler=scaler,
#)


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval()

    all_preds = []
    all_targets = []

    for images, clinical_vectors, labels in loader:
        images = images.to(device, non_blocking=True)
        clinical_vectors = clinical_vectors.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            logits = model(images, clinical_vectors)

        preds = torch.argmax(logits, dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_targets.extend(labels.numpy())

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)

    return all_targets, all_preds


def evaluate_classification_model(model, loader, device, idx_to_class):
    y_true, y_pred = get_predictions(model, loader, device)

    acc = accuracy_score(y_true, y_pred)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    labels = sorted(idx_to_class.keys())
    target_names = [idx_to_class[i] for i in labels]

    cm = confusion_matrix(y_true, y_pred, labels=labels)

    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=target_names,
        zero_division=0,
    )

    results = {
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "confusion_matrix": cm,
        "classification_report": report,
        "y_true": y_true,
        "y_pred": y_pred,
    }

    return results


In [ ]:
idx_to_class = {v: k for k, v in class_to_idx.items()}

results = evaluate_classification_model(
    #model=train_full_model,
    model=full_model,
    loader=data_module["loaders"]["test"],
    device=device,
    idx_to_class=idx_to_class,
)

print(f"Accuracy           : {results['accuracy']:.4f}")
print(f"Precision macro    : {results['precision_macro']:.4f}")
print(f"Recall macro       : {results['recall_macro']:.4f}")
print(f"F1 macro           : {results['f1_macro']:.4f}")
print(f"Precision weighted : {results['precision_weighted']:.4f}")
print(f"Recall weighted    : {results['recall_weighted']:.4f}")
print(f"F1 weighted        : {results['f1_weighted']:.4f}")

print("\nClassification report:\n")
print(results["classification_report"])

print("Confusion matrix:\n")
print(results["confusion_matrix"])


In [ ]:
import matplotlib.pyplot as plt

def plot_confusion_matrix(cm, idx_to_class):
    labels = [idx_to_class[i] for i in sorted(idx_to_class.keys())]

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm)

    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))

    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)

    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title("Confusion Matrix")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center")

    plt.tight_layout()
    plt.show()

plot_confusion_matrix(results["confusion_matrix"], idx_to_class)


# Combinations of the Columns

In [ ]:
from itertools import combinations

clinical_columns = [
    "Dx Periapical",
    "Sensibilidad",
    "Percusión",
    "Movilidad",
    "Palpación",
    "OD",
]

clinical_combinations = []

for r in range(1, len(clinical_columns) + 1):
    clinical_combinations.extend(combinations(clinical_columns, r))

clinical_combinations = [tuple(c) for c in clinical_combinations]

print("Total combinaciones:", len(clinical_combinations))

for i, combo in enumerate(clinical_combinations, start=1):
    print(i, combo)


In [ ]:
def build_clinical_vector_by_columns(row, selected_columns):
    vector = []

    # Dx Periapical
    dx_map = {
        "Periodontitis apical asintomática": [1, 0, 0, 0, 0],
        "Absceso apical crónico":            [0, 1, 0, 0, 0],
        "Periodontitis apical sintomática":  [0, 0, 1, 0, 0],
        "Tejidos apicales normales":         [0, 0, 0, 1, 0],
        "Absceso apical agudo":              [0, 0, 0, 0, 1],
    }

    # Binarias
    bin_map = {
        "positiva": 1.0,
        "positivo": 1.0,
        "negativa": 0.0,
        "negativo": 0.0,
    }

    # Movilidad
    movilidad_map = {
        "No valorable": [0, 0, 0],
        "1":            [1, 0, 0],
        "2":            [0, 1, 0],
        "3":            [0, 0, 1],
    }

    for column in selected_columns:

        if column == "Dx Periapical":
            value = str(row["Dx Periapical"]).strip()

            if value not in dx_map:
                raise ValueError(f"Dx Periapical no válido: {value}")

            vector.extend(dx_map[value])

        elif column == "Sensibilidad":
            value = str(row["Sensibilidad"]).strip().lower()

            if value not in bin_map:
                raise ValueError(f"Sensibilidad no válida: {value}")

            vector.append(bin_map[value])

        elif column == "Percusión":
            value = str(row["Percusión"]).strip().lower()

            if value not in bin_map:
                raise ValueError(f"Percusión no válida: {value}")

            vector.append(bin_map[value])

        elif column == "Movilidad":
            value = str(row["Movilidad"]).strip()

            if value not in movilidad_map:
                raise ValueError(f"Movilidad no válida: {value}")

            vector.extend(movilidad_map[value])

        elif column == "Palpación":
            value = str(row["Palpación"]).strip().lower()

            if value not in bin_map:
                raise ValueError(f"Palpación no válida: {value}")

            vector.append(bin_map[value])

        elif column == "OD":
            value = str(row["OD"]).strip()

            try:
                value = float(value)
            except ValueError:
                raise ValueError(f"OD no válido: {value}")

            vector.append(value)

        else:
            raise ValueError(f"Columna clínica no reconocida: {column}")

    return torch.tensor(vector, dtype=torch.float32)


In [ ]:
def create_samples_for_combination(selected_columns):
    combo_samples = []
    combo_errors = []

    for idx, row in df.iterrows():
        try:
            image_key = normalize_image_key(row["external_id"])

            if image_key not in image_path_dict:
                raise KeyError(f"No se encontró imagen para: {image_key}")

            clinical_vector = build_clinical_vector_by_columns(
                row=row,
                selected_columns=selected_columns,
            )

            if row["Clase"] not in class_to_idx:
                raise KeyError(f"Clase no válida: {row['Clase']}")

            label = class_to_idx[row["Clase"]]

            combo_samples.append({
                "image_key": image_key,
                "image_path": image_path_dict[image_key],
                "clinical_vector": clinical_vector,
                "label": label,
            })

        except Exception as e:
            combo_errors.append({
                "row_idx": idx,
                "id": row.get("id", None),
                "external_id": row.get("external_id", None),
                "clase": row.get("Clase", None),
                "selected_columns": selected_columns,
                "error": str(e),
            })

    return combo_samples, combo_errors


In [ ]:
test_combo = (
    "Dx Periapical",
    "Sensibilidad",
    "Movilidad",
    "Palpación",
)

combo_samples, combo_errors = create_samples_for_combination(test_combo)

print("Combinación:", test_combo)
print("Samples válidos:", len(combo_samples))
print("Errores:", len(combo_errors))
print("Tamaño vector clínico:", len(combo_samples[0]["clinical_vector"]))

if combo_errors:
    print(combo_errors[:5])


In [ ]:
test_combo = ("OD",)

combo_samples, combo_errors = create_samples_for_combination(test_combo)

print("Combinación:", test_combo)
print("Samples válidos:", len(combo_samples))
print("Errores:", len(combo_errors))
print("Tamaño vector clínico:", len(combo_samples[0]["clinical_vector"]))


In [ ]:
import copy
import pandas as pd

def build_model_for_clinical_dim(clinical_dim):
    embedding_dim = 18432
    input_dim = embedding_dim + clinical_dim

    fresh_extractor = copy.deepcopy(extractor)

    fused_classifier = FusedClassifier(
        input_dim=input_dim,
        num_classes=num_classes,
    )

    model = FullMultimodalDentalDeppGAModel(
        embedding_extractor=fresh_extractor,
        fused_classifier=fused_classifier,
    ).to(device)

    return model


In [ ]:
combination_results = {}

def run_combination_experiment(
    selected_columns,
    batch_size=64,
    epochs=50,
    val_split=0.15,
    test_split=0.15,
    seed=42,
):
    print("\n" + "=" * 80)
    print("Combinación:", selected_columns)
    print("=" * 80)

    combo_samples, combo_errors = create_samples_for_combination(selected_columns)

    if len(combo_samples) == 0:
        raise ValueError(f"No hay samples válidos para {selected_columns}")

    clinical_dim = len(combo_samples[0]["clinical_vector"])
    print("Samples:", len(combo_samples))
    print("Errores:", len(combo_errors))
    print("Clinical dim:", clinical_dim)

    combo_data_module = load_data_module(
        samples=combo_samples,
        batch_size=batch_size,
        val_split=val_split,
        test_split=test_split,
        seed=seed,
        num_workers=4,
    )

    model = build_model_for_clinical_dim(clinical_dim)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW([
        {"params": model.embedding_extractor.parameters(), "lr": 1e-6},
        {"params": model.fused_classifier.parameters(), "lr": 3e-4},
    ], weight_decay=5e-4)

    # AMP scaler por experimento
    exp_scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

    trained_model, history = train_model(
        model=model,
        train_loader=combo_data_module["loaders"]["train"],
        val_loader=combo_data_module["loaders"]["val"],
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        epochs=epochs,
        scaler=exp_scaler,
    )

    results = evaluate_classification_model(
        model=trained_model,
        loader=combo_data_module["loaders"]["test"],
        device=device,
        idx_to_class=idx_to_class,
    )

    print("\nAccuracy:", results["accuracy"])
    print("F1 macro:", results["f1_macro"])
    print("\nClassification report:")
    print(results["classification_report"])

    plot_confusion_matrix(results["confusion_matrix"], idx_to_class)

    key = " + ".join(selected_columns)

    combination_results[key] = {
        "columns": selected_columns,
        "clinical_dim": clinical_dim,
        "samples_count": len(combo_samples),
        "errors": combo_errors,
        "data_module": combo_data_module,
        "model": trained_model,
        "history": history,
        "metrics": results,
    }

    return combination_results[key]


In [ ]:
all_combination_results = {}
failed_combination_results = {}


In [ ]:
def run_all_combination_experiments(
    clinical_combinations,
    batch_size=64,
    epochs=50,
    val_split=0.15,
    test_split=0.15,
    seed=42,
):
    global all_combination_results
    global failed_combination_results

    total = len(clinical_combinations)

    for i, combo in enumerate(clinical_combinations, start=1):
        key = " + ".join(combo)

        print("\n\n" + "#" * 100)
        print(f"EXPERIMENTO {i}/{total}")
        print(f"Combinación: {key}")
        print("#" * 100)

        try:
            result = run_combination_experiment(
                selected_columns=combo,
                batch_size=batch_size,
                epochs=epochs,
                val_split=val_split,
                test_split=test_split,
                seed=seed,
            )

            all_combination_results[key] = result

        except Exception as e:
            print(f"❌ Error en combinación: {key}")
            print(e)

            failed_combination_results[key] = {
                "columns": combo,
                "error": str(e),
            }

        # Limpiar caché CUDA entre experimentos para liberar VRAM
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("\n" + "=" * 100)
    print("EXPERIMENTOS TERMINADOS")
    print("=" * 100)
    print("Combinaciones exitosas:", len(all_combination_results))
    print("Combinaciones con error:", len(failed_combination_results))

    return all_combination_results


In [ ]:
all_results = run_all_combination_experiments(
    clinical_combinations=clinical_combinations,
    batch_size=64,
    epochs=50,
    val_split=0.15,
    test_split=0.15,
    seed=42,
)


In [ ]:
def build_combination_summary_table(results_dict):
    rows = []

    for key, result in results_dict.items():
        metrics = result["metrics"]

        rows.append({
            "combination": key,
            "num_columns": len(result["columns"]),
            "clinical_dim": result["clinical_dim"],
            "samples_count": result["samples_count"],
            "accuracy": metrics["accuracy"],
            "f1_macro": metrics["f1_macro"],
        })

    summary_df = pd.DataFrame(rows)

    summary_df = summary_df.sort_values(
        by=["f1_macro", "accuracy"],
        ascending=False,
    ).reset_index(drop=True)

    summary_df.insert(0, "rank", range(1, len(summary_df) + 1))

    return summary_df


In [ ]:
summary_df = build_combination_summary_table(all_combination_results)

summary_df

best_row = summary_df.iloc[0]

print("🏆 Mejor combinación")
print("Combinación:", best_row["combination"])
print("Columnas usadas:", best_row["num_columns"])
print("Clinical dim:", best_row["clinical_dim"])
print("Accuracy:", best_row["accuracy"])
print("F1 macro:", best_row["f1_macro"])
